# Assignment 9

In [22]:
import cv2
import mediapipe as mp
import pandas as pd
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


def extract_mediapipe_to_csv(data_path: str, save_path: str, model_path: str):

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]
    static = not any(data_path.lower().endswith(ext) for ext in video_extensions)

    base_options = python.BaseOptions(model_asset_path=model_path)

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,  
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    data = []

    if static:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            image = cv2.imread(data_path)
            if image is None:
                raise ValueError(f"Could not read image: {data_path}")

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            results = landmarker.detect(mp_image)

            if results.pose_landmarks:
                row = {"FrameNo": 0}

                for joint in JOINT_ORDER:
                    idx = LANDMARK_INDEX[joint]
                    lm = results.pose_landmarks[0][idx]

                    row[f"{joint}_x"] = lm.x
                    row[f"{joint}_y"] = lm.y
                    row[f"{joint}_z"] = lm.z

                data.append(row)

    else:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            cap = cv2.VideoCapture(data_path)

            if not cap.isOpened():
                raise ValueError(f"Could not open video: {data_path}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                timestamp_ms = int((frame_idx / fps) * 1000)
                results = landmarker.detect_for_video(mp_image, timestamp_ms)

                if results.pose_landmarks:
                    row = {"FrameNo": frame_idx}

                    for joint in JOINT_ORDER:
                        idx = LANDMARK_INDEX[joint]
                        lm = results.pose_landmarks[0][idx]

                        row[f"{joint}_x"] = lm.x
                        row[f"{joint}_y"] = lm.y
                        row[f"{joint}_z"] = lm.z

                    data.append(row)

                frame_idx += 1

            cap.release()

    # ---- SAVE CSV ----
    columns = ["FrameNo"]
    for joint in JOINT_ORDER:
        columns += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    df = pd.DataFrame(data)
    df = df[columns]

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)

    print(f"Saved {len(df)} rows to {save_path}")

In [53]:
extract_mediapipe_to_csv(
    data_path="../data/still_video.mov",
    save_path="../data/still_video_mediapipe.csv",
    model_path="../data/pose_landmarker.task"
)

I0000 00:00:1776674438.895264 9209142 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1776674439.140993 9494101 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776674439.256478 9494105 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 105 rows to ../data/still_video_mediapipe.csv


### Joint distance Enis

In [ ]:
shoulder_to_shoulder = 0.036
Shoulder_to_hip = 0.55
hip_to_hip = 0.25
hip_to_knee = 0.47
knee_to_ankle = 0.43

### Save video to csv

In [ ]:
""""
video_folder = "../data"
model_path = "../data/pose_landmarker.task"
output_folder = "../data"

video_extensions = [".mp4", ".mov", ".avi", ".mkv"]

for file_name in os.listdir(video_folder):
    
    if not any(file_name.lower().endswith(ext) for ext in video_extensions):
        continue

    video_path = os.path.join(video_folder, file_name)

    base_name = os.path.splitext(file_name)[0]
    save_name = f"{base_name}_mediapipe.csv"
    save_path = os.path.join(output_folder, save_name)

    print(f"Processing: {file_name}")

    extract_mediapipe_to_csv(
        data_path=video_path,
        save_path=save_path,
        model_path=model_path
    )
print("Done.")
"""

### Jokobs code for comparing mediapipe to ground truth

In [ ]:
import numpy as np

# Load your mediapipe CSV
df = pd.read_csv("../data/still_video_mediapipe.csv")

def euclidean(x1, y1, z1, x2, y2, z2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2 + (z1-z2)**2)

# Calculate distances per frame
df["hip_to_shoulder"]      = euclidean(df["left_hip_x"], df["left_hip_y"], df["left_hip_z"],
                                        df["left_shoulder_x"], df["left_shoulder_y"], df["left_shoulder_z"])
df["knee_to_hip"]          = euclidean(df["left_knee_x"], df["left_knee_y"], df["left_knee_z"],
                                        df["left_hip_x"], df["left_hip_y"], df["left_hip_z"])
df["hip_to_hip"]           = euclidean(df["left_hip_x"], df["left_hip_y"], df["left_hip_z"],
                                        df["right_hip_x"], df["right_hip_y"], df["right_hip_z"])
df["knee_to_ankle"]        = euclidean(df["left_knee_x"], df["left_knee_y"], df["left_knee_z"],
                                        df["left_foot_x"], df["left_foot_y"], df["left_foot_z"])
df["shoulder_to_shoulder"] = euclidean(df["left_shoulder_x"], df["left_shoulder_y"], df["left_shoulder_z"],
                                        df["right_shoulder_x"], df["right_shoulder_y"], df["right_shoulder_z"])

# Average across frames
avg = df[["hip_to_shoulder", "knee_to_hip", "hip_to_hip", "knee_to_ankle", "shoulder_to_shoulder"]].mean()

# Ground truth
ground_truth = {
    "hip_to_shoulder":      55,
    "knee_to_hip":          47,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}


# Scale using hip_to_shoulder as reference
scale = ground_truth["hip_to_shoulder"] / avg["hip_to_shoulder"]
print(f"Scale factor: {scale:.2f} cm per unit\n")

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key] * scale
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")

Scale factor: 321.55 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 55.0            55.0          0.0        0.0%
knee_to_hip                     47.0            42.9          4.1        8.6%
hip_to_hip                      25.0            35.4         10.4       41.5%
knee_to_ankle                   43.0            41.7          1.3        3.0%
shoulder_to_shoulder            38.0            56.7         18.7       49.1%


In [ ]:
import cv2

video_path = "../data/still_video.mov"
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

print(f"Width: {width}px, Height: {height}px")

Width: 1920px, Height: 1080px


In [ ]:
import numpy as np

# Load your mediapipe CSV
#df = pd.read_csv("../data/1m_step_low_mediapipe.csv")
df = pd.read_csv("../data/still_video_mediapipe.csv")


FRAME_WIDTH = 1920   
FRAME_HEIGHT = 1080  

# Convert normalized coords to pixels
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * FRAME_WIDTH
    df[f"{joint}_y"] = df[f"{joint}_y"] * FRAME_HEIGHT


def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

# Calculate distances per frame
df["hip_to_shoulder"]        = euclidean(df["left_hip_x"], df["left_hip_y"],
                                    df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]            = euclidean(df["left_knee_x"], df["left_knee_y"],
                                    df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]             = euclidean(df["left_hip_x"], df["left_hip_y"],
                                    df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                    df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"]   = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                    df["right_shoulder_x"], df["right_shoulder_y"])

# Average across frames
avg = df[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip",
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].mean()

# Ground truth
ground_truth = {
    "hip_to_shoulder":      55,
    "knee_to_hip":          47,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}


# Scale using hip_to_shoulder as reference
scale = ground_truth["hip_to_shoulder"] / avg["hip_to_shoulder"]
print(f"Scale factor: {scale:.2f} cm per unit\n")

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key] * scale
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")

Scale factor: 0.18 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 55.0            55.0          0.0        0.0%
knee_to_hip                     47.0            46.3          0.7        1.6%
hip_to_hip                      25.0            21.6          3.4       13.4%
knee_to_ankle                   43.0            39.0          4.0        9.3%
shoulder_to_shoulder            38.0            34.7          3.3        8.7%
